## Summary: GAN Architecture & Key Insights

### **Core Components Overview**

```
┌─────────────────────────────────────────────────────────────┐
│                    GAN ARCHITECTURE                         │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  GENERATOR (Creates Fake Images)                            │
│  ═══════════════════════════════════════════                │
│  • Input: Random noise Z ~ N(0,1)                           │
│  • Dense layers → Feature expansion                         │
│  • Reshape → Spatial structure                              │
│  • Conv2DTranspose → Progressive upsampling                 │
│  • Output: 28×28×1 image with Tanh [-1,1]                  │
│                                                              │
│                         ↓                                    │
│                                                              │
│  DISCRIMINATOR (Classifies Real/Fake)                       │
│  ════════════════════════════════════════════               │
│  • Input: Image (real or fake)                              │
│  • Conv2D → Feature extraction & downsampling               │
│  • LeakyReLU → Gradient flow                                │
│  • Flatten → Convert to vector                              │
│  • Dense layers → Binary classification                     │
│  • Output: Single value [0,1] via Sigmoid                   │
│                         │                                    │
│                         ↓                                    │
│              Real (1) or Fake (0)?                          │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

### **Training Dynamics - The Adversarial Process**

#### **Equilibrium State:**
- **Perfect Generator:** Produces images indistinguishable from real data
- **Perfect Discriminator:** Cannot distinguish real from fake
- Both networks learn complementary abilities

#### **Loss Perspective:**
- **Discriminator Loss:** $L_D = -E[\log(D(x))] - E[\log(1-D(G(z)))]$
  - Maximize ability to correctly classify
  - Penalized when it misclassifies real as fake or vice versa
  
- **Generator Loss:** $L_G = -E[\log(D(G(z)))]$
  - Maximize discriminator's probability of misclassification
  - Wants discriminator to output ~1 for fake images

### **Why These Specific Design Choices?**

#### **Generator: ReLU + Tanh**
- **ReLU (hidden):** Provides strong non-linearity for complex transformations
- **Tanh (output):** Creates smooth, continuous transitions between -1 and 1
- **Result:** Smooth fake image generation

#### **Discriminator: LeakyReLU**
- **Problem it solves:** Standard ReLU can cause "dead neurons"
- **LeakyReLU ($\alpha=0.2$):** Allows small negative gradients
- **Result:** Stable gradient flow during adversarial competition

#### **Batch Normalization**
- **Effect:** Stabilizes training by reducing covariate shift
- **In Generator:** Helps propagate gradients through dense layers
- **In Discriminator:** Prevents extreme value oscillations

#### **Dropout in Discriminator**
- **Purpose:** Regularization and prevents discriminator overfitting
- **Side effect:** Forces discriminator to learn robust features
- **Better competitiveness:** Less overconfident discriminator means generator learns real features

### **Common GAN Issues & Solutions**

| Issue | Cause | Solution |
|-------|-------|----------|
| Mode Collapse | Generator only produces subset of variations | Use spectral norm, minibatch discrimination, Wasserstein loss |
| Vanishing Gradients | Generator loss saturates | Use Wasserstein loss instead of BCE |
| Oscillating Loss | Training instability | Adjust learning rates, use gradient penalties |
| Poor Quality | Insufficient training | Train longer, increase model capacity |

### **Key Takeaways**

✓ **Generator:** Learns to map random latent space → realistic image space
✓ **Discriminator:** Learns to distinguish real from synthetic distributions
✓ **Adversarial Training:** Two networks in competition drive each other to excellence
✓ **Architecture Matters:** ConvTranspose for upsampling, Conv for downsampling
✓ **Activations Matter:** LeakyReLU enables better gradient flow than ReLU
✓ **Normalization:** Batch norm + Dropout stabilize training significantly

### **Advanced Directions**

- **Conditional GAN (cGAN):** Control image generation with labels
- **Wasserstein GAN (WGAN):** Better loss function for stable training
- **StyleGAN:** Control image features at different scales
- **Progressive GAN:** Gradually increase resolution during training
- **Diffusion Models:** Modern alternative to GANs for generation

In [ ]:
# Compare Real vs Generated Images
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Real vs Generated Images Comparison', fontsize=16, fontweight='bold')

# Real images (from first batch)
real_batch = next(iter(dataset))[:8]
real_batch_normalized = (real_batch + 1) / 2  # Denormalize to [0, 1]

# Generated images
generated_batch = generate_images(generator, num_images=8)

# Plot real images
for idx, ax in enumerate(axes[0]):
    img = real_batch_normalized[idx].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_title(f'Real {idx+1}', fontsize=10)

# Plot generated images
for idx, ax in enumerate(axes[1]):
    img = generated_batch[idx].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_title(f'Generated {idx+1}', fontsize=10)

plt.tight_layout()
plt.show()

print("✓ Comparison complete!")

In [ ]:
# Plot training loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: All losses together
axes[0].plot(gen_losses, label='Generator Loss', linewidth=2)
axes[0].plot(disc_losses_real, label='Discriminator Loss (Real)', linewidth=2)
axes[0].plot(disc_losses_fake, label='Discriminator Loss (Fake)', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('GAN Training Losses', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Discriminator average loss vs Generator loss
avg_disc_loss = [(r + f) / 2 for r, f in zip(disc_losses_real, disc_losses_fake)]
axes[1].plot(gen_losses, label='Generator Loss', linewidth=2, marker='o')
axes[1].plot(avg_disc_loss, label='Discriminator Avg Loss', linewidth=2, marker='s')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Generator vs Discriminator Performance', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 60)
print("TRAINING STATISTICS")
print("=" * 60)
print(f"Final Generator Loss: {gen_losses[-1]:.4f}")
print(f"Final Discriminator Loss (Real): {disc_losses_real[-1]:.4f}")
print(f"Final Discriminator Loss (Fake): {disc_losses_fake[-1]:.4f}")
print(f"Average Generator Loss: {np.mean(gen_losses):.4f}")
print(f"Average Discriminator Loss: {np.mean(avg_disc_loss):.4f}")

## 10. Evaluate Model Performance

### **Evaluation Metrics:**

1. **Loss Convergence:** Examine discriminator and generator loss trends
2. **Visual Quality:** Assess generated image coherence
3. **Diversity:** Check if generator produces varied outputs
4. **Mode Coverage:** Ensure generator covers different digit classes

In [ ]:
def generate_images(generator, num_images=16, latent_dim=100):
    """Generate synthetic images from random noise."""
    noise = np.random.normal(0, 1, (num_images, latent_dim))
    generated_images = generator.predict(noise, verbose=0)
    
    # Denormalize from [-1, 1] to [0, 1]
    generated_images = (generated_images + 1) / 2
    
    return generated_images

# Generate 16 synthetic images
generated_images = generate_images(generator, num_images=16)

# Visualize generated images
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('GAN Generated Images', fontsize=16, fontweight='bold')

for idx, ax in enumerate(axes.flat):
    img = generated_images[idx].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_title(f'Generated {idx+1}')

plt.tight_layout()
plt.show()

print("✓ Generated synthetic digit images successfully!")

## 9. Generate and Visualize Images

Now we'll use the trained generator to create synthetic images from random noise vectors.

### **Image Generation Process:**
1. Sample random noise from normal distribution
2. Pass through generator network
3. Denormalize from [-1, 1] to [0, 1] for display
4. Visualize results

In [ ]:
def train_gan(generator, discriminator, gan, dataset, epochs=50):
    """
    Custom training loop for GAN.
    """
    latent_dim = 100
    
    # Storage for losses
    gen_losses = []
    disc_losses_real = []
    disc_losses_fake = []
    
    for epoch in range(epochs):
        epoch_gen_loss = 0
        epoch_disc_loss_real = 0
        epoch_disc_loss_fake = 0
        batch_count = 0
        
        for real_images in dataset:
            batch_size = real_images.shape[0]
            
            # ========== DISCRIMINATOR TRAINING ==========
            # Enable discriminator training
            discriminator.trainable = True
            
            # Generate fake images
            noise = np.random.normal(0, 1, (batch_size, latent_dim))
            fake_images = generator.predict(noise, verbose=0)
            
            # Train discriminator on real images
            disc_loss_real = discriminator.train_on_batch(
                real_images, 
                np.ones((batch_size, 1))  # Label: all real (1)
            )
            
            # Train discriminator on fake images
            disc_loss_fake = discriminator.train_on_batch(
                fake_images,
                np.zeros((batch_size, 1))  # Label: all fake (0)
            )
            
            # ========== GENERATOR TRAINING ==========
            # Freeze discriminator
            discriminator.trainable = False
            
            # Generate new noise
            noise = np.random.normal(0, 1, (batch_size, latent_dim))
            
            # Train generator (fool discriminator)
            gen_loss = gan.train_on_batch(
                noise,
                np.ones((batch_size, 1))  # We want discriminator to think these are real
            )
            
            # Accumulate losses
            epoch_gen_loss += gen_loss
            epoch_disc_loss_real += disc_loss_real
            epoch_disc_loss_fake += disc_loss_fake
            batch_count += 1
        
        # Average losses over epoch
        avg_gen_loss = epoch_gen_loss / batch_count
        avg_disc_loss_real = epoch_disc_loss_real / batch_count
        avg_disc_loss_fake = epoch_disc_loss_fake / batch_count
        
        gen_losses.append(avg_gen_loss)
        disc_losses_real.append(avg_disc_loss_real)
        disc_losses_fake.append(avg_disc_loss_fake)
        
        # Print progress
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}")
            print(f"  Generator Loss: {avg_gen_loss:.4f}")
            print(f"  Disc Loss (Real): {avg_disc_loss_real:.4f}")
            print(f"  Disc Loss (Fake): {avg_disc_loss_fake:.4f}")
            print()
    
    return gen_losses, disc_losses_real, disc_losses_fake

# Train the GAN (using fewer epochs for demonstration)
print("Starting GAN Training...\n")
gen_losses, disc_losses_real, disc_losses_fake = train_gan(
    generator, discriminator, gan, dataset, epochs=25
)
print("Training completed!")

## 8. Train the GAN

We'll implement a custom training loop to alternately train the discriminator and generator.

### **Training Loop Logic:**

```
For each epoch:
    For each batch of real images:
        1. Create fake images from generator
        2. Train discriminator on real images (target=1)
        3. Train discriminator on fake images (target=0)
        4. Train generator through frozen discriminator (target=1)
        
Monitor:
    - Discriminator loss on real images
    - Discriminator loss on fake images
    - Generator loss
```

In [ ]:
# Compile Discriminator
discriminator.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Build Combined GAN Model
def build_gan(generator, discriminator):
    """
    Builds the combined GAN model.
    Generator tries to fool discriminator.
    """
    # Freeze discriminator weights during GAN training
    discriminator.trainable = False
    
    model = models.Sequential([
        generator,
        discriminator
    ])
    
    return model

gan = build_gan(generator, discriminator)

# Compile GAN
gan.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    loss='binary_crossentropy'
)

print("=" * 50)
print("DISCRIMINATOR CONFIGURATION")
print("=" * 50)
print(f"Trainable: {discriminator.trainable}")
print(f"Optimizer: {discriminator.optimizer.__class__.__name__}")

print("\n" + "=" * 50)
print("GAN CONFIGURATION")
print("=" * 50)
print(f"Discriminator Trainable: {discriminator.trainable}")
print(f"Generator Trainable: {generator.trainable}")
print(f"Optimizer: {gan.optimizer.__class__.__name__}")
print(f"Loss Function: Binary Crossentropy")

## 7. Compile GAN Model

We now create a combined GAN model and set up the training configuration.

### **GAN Training Strategy:**

The GAN model consists of:
1. **Discriminator** (trained separately)
2. **Frozen Generator** + **Discriminator** (combined model)

**Training Process:**
- **Step 1:** Train Discriminator on real batches → Loss should go down
- **Step 2:** Train Discriminator on fake batches → Loss should go down
- **Step 3:** Train Combined Model (Generator) → Generator loss should go down, Discriminator loss goes up
  - Generator wants to fool discriminator
  - Discriminator frozen during this step

## 6. Explain Discriminator Components

### **Component Breakdown:**

#### **Input Layer**
- **Shape:** (batch_size, 28, 28, 1)
- **Purpose:** Accepts both real and fake images
- **Preprocessing:** Already normalized to [-1, 1] range

#### **Conv2D Layers (Feature Extraction)**
- **Purpose:** Extract spatial features and hierarchical patterns
- **Layer 1:** 32 filters, 3x3 kernel, stride=2 → (14, 14, 32)
  - Detects low-level features: edges, textures
- **Layer 2:** 64 filters, 3x3 kernel, stride=2 → (7, 7, 64)
  - Detects intermediate features: shapes, patterns
- **Layer 3:** 128 filters, 3x3 kernel, stride=2 → (4, 4, 128)
  - Detects high-level semantic features
- **Layer 4:** 256 filters, 3x3 kernel, stride=1 → (4, 4, 256)
  - Further refinement without spatial reduction

#### **Stride and Downsampling**
- **Stride=2:** Reduces spatial dimensions by half with each layer
- **Purpose:** Progressive downsampling while increasing channel depth
- **Progressive pathway:** (28,28) → (14,14) → (7,7) → (4,4)

#### **Dropout Layers**
- **Purpose:** Regularization to prevent overfitting
- **Rate:** 0.3 (drop 30% of neurons randomly during training)
- **Effect:** Prevents co-adaptation of neurons
- **Usage:** Between dense layers and after convolutional blocks

#### **LeakyReLU Activation**
- **Formula:** $f(x) = \begin{cases} x & \text{if } x > 0 \\ 0.2x & \text{if } x \leq 0 \end{cases}$
- **Alpha:** 0.2 (slope for negative values)
- **Why LeakyReLU over ReLU:**
  - Standard ReLU can cause "dying ReLU" problem (neurons output 0 forever)
  - Leaky ReLU allows small negative gradients to flow
  - Better convergence in adversarial training
  - More stable for discriminator training

#### **Batch Normalization**
- Applied after Conv2D layers (often not on first layer)
- Stabilizes discriminator training
- Prevents extreme value shifts during adversarial updates
- Improves gradient flow

#### **Flatten Layer**
- **Input:** (batch_size, 4, 4, 256)
- **Output:** (batch_size, 4096)
- **Purpose:** Converts spatial feature maps to 1D vector
- **Critical:** Bridges convolutional feature extraction to dense classification

#### **Dense Classification Layers**
- **Layer 1:** 512 neurons with LeakyReLU and Dropout
- **Layer 2:** 256 neurons with LeakyReLU and Dropout
- **Purpose:** Learn non-linear decision boundaries for classification

#### **Output Layer**
- **Neurons:** 1
- **Activation:** **Sigmoid** → Output in [0, 1] range
- **Interpretation:**
  - Output ≈ 1: Classified as REAL image
  - Output ≈ 0: Classified as FAKE image
- **Loss Function:** Binary Crossentropy

### **Comparison: Why LeakyReLU for Discriminator, ReLU for Generator?**

| Aspect | Generator | Discriminator |
|--------|-----------|---------------|
| Activation | ReLU | LeakyReLU |
| Purpose | Create smooth, continuous output | Make sharp distinctions |
| Problem | None critical | Dying ReLU happens frequently |
| Training Dynamics | Stable upsampling | Needs gradient flow from all inputs |

In [ ]:
def build_discriminator():
    """
    Builds the discriminator network.
    Input: image (batch_size, 28, 28, 1)
    Output: binary classification (batch_size, 1) - Real (1) or Fake (0)
    """
    model = models.Sequential([
        # Layer 1: Conv2D - Feature extraction from input
        # No batch norm in first layer (often omitted in discriminators)
        layers.Conv2D(32, kernel_size=3, strides=2, padding='same', 
                     input_shape=(28, 28, 1)),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),
        
        # Layer 2: Conv2D - Further feature extraction
        layers.Conv2D(64, kernel_size=3, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),
        
        # Layer 3: Conv2D - Refined feature maps
        layers.Conv2D(128, kernel_size=3, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),
        
        # Layer 4: Conv2D - Final feature extraction
        layers.Conv2D(256, kernel_size=3, strides=1, padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),
        
        # Layer 5: Flatten - Convert spatial features to vector
        layers.Flatten(),
        
        # Layer 6: Dense - Classification layers
        layers.Dense(512),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),
        
        layers.Dense(256),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),
        
        # Output layer: Binary classification
        layers.Dense(1, activation='sigmoid')
    ])
    
    return model

# Create discriminator
discriminator = build_discriminator()
discriminator.summary()

## 5. Build the Discriminator Network

The Discriminator acts as a binary classifier that distinguishes real images from fake (generated) images.

**Architecture Overview:**
```
Input Image (28, 28, 1)
  ↓
Conv2D (Feature Extraction)
  ↓
Conv2D (Feature Extraction)
  ↓
MaxPooling (Downsampling)
  ↓
Conv2D (Feature Refinement)
  ↓
Flatten
  ↓
Dense Layers
  ↓
Output (Binary Classification)
```

## 4. Explain Generator Components

### **Component Breakdown:**

#### **Input Layer (Latent Vector)**
- **Purpose:** Entry point for random noise
- **Shape:** (batch_size, 100) - 100-dimensional random vector
- **Role:** Captures the randomness factor; different noise vectors produce different outputs
- **Sampling:** Generated from uniform or normal distribution

#### **Dense Layers (Layers 1-2)**
- **Purpose:** Color expansion and feature mapping
- **Layer 1:** 256 neurons - Expand small latent vector to richer feature representation
- **Layer 2:** 512 neurons - Further expansion to prepare for spatial structure
- **Activation:** ReLU (Rectified Linear Unit) to introduce non-linearity
- **Batch Normalization:** Stabilizes training by normalizing layer inputs

#### **Reshape Layer**
- **Input:** (batch_size, 512)
- **Output:** (batch_size, 4, 4, 32)
- **Purpose:** Converts flat vector into spatial feature maps (4x4 spatial dimensions, 32 channels)
- **Critical:** Bridges dense layers to convolutional layers

#### **Conv2DTranspose Layers (Upsampling)**
- **Purpose:** Progressively upsample feature maps to increase spatial resolution
- **Layer 4:** (4, 4, 32) → (7, 7, 128) with 5x5 kernel, stride=2
- **Layer 5:** (7, 7, 128) → (14, 14, 64) with 5x5 kernel, stride=2
- **Layer 6:** (14, 14, 64) → (28, 28, 32) with 5x5 kernel, stride=2
- **Kernel Size:** Larger kernels (5x5) capture more context during upsampling
- **Stride=2:** Doubles spatial dimensions with each layer

#### **Output Layer**
- **Kernel:** 1x1 convolution (often 3x3 for detail refinement)
- **Channels:** 1 (grayscale) or 3 (RGB for color images)
- **Activation:** **Tanh** - Outputs values in [-1, 1] range, matching normalized input data
- **Why Tanh:** Symmetric activation around zero, better for normalized data than ReLU

#### **Activation Functions Summary:**
- **ReLU (Hidden layers):** $f(x) = max(0, x)$ - Introduces non-linearity, prevents gradient vanishing
- **Tanh (Output layer):** $f(x) = \frac{e^{2x} - 1}{e^{2x} + 1}$ - Maps output to [-1, 1]

#### **Batch Normalization Role:**
- Normalizes layer inputs to zero mean and unit variance
- Accelerates training convergence
- Reduces sensitivity to network weight initialization
- Allows higher learning rates

In [ ]:
def build_generator():
    """
    Builds the generator network.
    Input: random noise vector (batch_size, latent_dim)
    Output: synthetic image (batch_size, 28, 28, 1)
    """
    latent_dim = 100
    
    model = models.Sequential([
        # Layer 1: Dense layer to expand latent vector
        layers.Dense(256, input_dim=latent_dim),
        layers.BatchNormalization(),
        layers.ReLU(),
        
        # Layer 2: Dense layer for further expansion
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.ReLU(),
        
        # Layer 3: Reshape to spatial dimensions
        # 512 features -> 4x4x32 = 512 features
        layers.Reshape((4, 4, 32)),
        
        # Layer 4: Conv2DTranspose - Upsample to 7x7
        layers.Conv2DTranspose(128, kernel_size=5, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        
        # Layer 5: Conv2DTranspose - Upsample to 14x14
        layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        
        # Layer 6: Conv2DTranspose - Upsample to 28x28
        layers.Conv2DTranspose(32, kernel_size=5, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        
        # Output layer: 1 channel, Tanh activation for [-1, 1] range
        layers.Conv2D(1, kernel_size=3, padding='same', activation='tanh')
    ])
    
    return model

# Create generator
generator = build_generator()
generator.summary()

## 3. Build the Generator Network

The Generator takes random noise as input and produces synthetic images that resemble real data.

**Architecture Overview:**
```
Input (latent vector: 100,) 
  ↓
Dense Layer (256 neurons)
  ↓
Reshape to (4, 4, 16)
  ↓
Conv2DTranspose (Upsampling)
  ↓
Conv2DTranspose (Upsampling)
  ↓
Conv2DTranspose (Output - 1 channel)
  ↓
Output (28, 28, 1)
```

In [ ]:
# Load MNIST dataset
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()

# Combine training and test data
x_data = np.concatenate([x_train, x_test], axis=0)

# Preprocess: Normalize to [-1, 1] range
x_data = x_data.astype(np.float32)
x_data = (x_data - 127.5) / 127.5  # Normalize to [-1, 1]

# Add channel dimension: (samples, 28, 28) -> (samples, 28, 28, 1)
x_data = np.expand_dims(x_data, axis=-1)

print(f"Dataset shape: {x_data.shape}")
print(f"Pixel value range: [{x_data.min():.2f}, {x_data.max():.2f}]")

# Create dataset pipeline
batch_size = 32
dataset = tf.data.Dataset.from_tensor_slices(x_data)
dataset = dataset.shuffle(buffer_size=10000)
dataset = dataset.batch(batch_size)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

print(f"Batch size: {batch_size}")
print(f"Total batches per epoch: {len(x_data) // batch_size}")

## 2. Load and Preprocess Dataset

We'll use MNIST dataset (handwritten digits) for this GAN model. The dataset contains 28x28 grayscale images.

**Data Processing Steps:**
- Load MNIST dataset
- Normalize pixel values to [-1, 1] range (important for GAN training with tanh activation)
- Reshape data to add channel dimension (28, 28, 1)
- Create batches for training

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 1. Import Required Libraries

# Generative Adversarial Network (GAN) for Image Generation

This notebook demonstrates how to build a complete GAN model to generate synthetic images. We'll explore the architecture, components, and training process of both the Generator and Discriminator networks.